# Day 070 · 质量因子
**Quality Factors** · 阶段 P3 · 数据与因子工程

> 这一节我们认识第三把尺子,也是最受巴菲特推崇的一把:质量因子。前两节价值挑便宜、成长挑长得快,可不管买便宜还是买成长,你最怕的都是买到一家「绣花枕头」公司——看着光鲜,其实不会赚钱。质量因子干的就是这件事:在一堆股票里,挑出那些真正能持续、稳定、高效赚钱的好公司。怎么衡量「会赚钱」?最常用的一把尺子叫净资产收益率,英文缩写ROE,说白了就是「1 块钱本钱,一年能赚回几毛」。但这里有个大坑:高ROE有真有假,有的是靠借一屁股债撑出来的,有的利润压根没收到现金、全是欠条。所以这一节还要教你一面「照妖镜」——用经营现金流去验真,把那些账面赚钱、口袋没钱的「纸面富贵」公司揪出来。

---

**课件生成日期:** 2026-06-27  ·  **建议学习时长:** 22 分钟

学习路径建议:1)先看视频建立直觉 → 2)阅读本 notebook 跑代码 → 3)看 PDF 课件复习要点 → 4)做自测题

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有需要的 Python 包,缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续下面的代码

> 这一格只在第一次跑要等几十秒,后面再开 notebook 就秒过。

In [1]:
# === 环境自检 + 自动安装(运行此单元格即可) ===
# 检测缺失的库 → 自动 pip 安装 → 注入中文字体 → 一行命令搞定
import importlib
import subprocess
import sys
import os

REQUIRED = ["baostock", "matplotlib", "numpy", "numpy_financial", "pandas", "scipy", "sklearn", "statsmodels", "yfinance"]
PIP_NAME = {
    "sklearn": "scikit-learn",
    "cv2": "opencv-python",
    "PIL": "Pillow",
    "bs4": "beautifulsoup4",
    "yaml": "PyYAML",
}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))

if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,正在自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置(让 matplotlib 不出乱码) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

CJK_FONT_PATHS = [
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",  # Linux/WSL
    "C:/Windows/Fonts/msyh.ttc",                               # Windows 微软雅黑
    "C:/Windows/Fonts/simhei.ttf",                             # Windows 黑体
    "/System/Library/Fonts/PingFang.ttc",                      # macOS 苹方
    "/System/Library/Fonts/STHeiti Medium.ttc",                # macOS 黑体
]
for p in CJK_FONT_PATHS:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "Microsoft YaHei",
                                    "PingFang SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪 — 现在可以跑下面的代码单元格")


✓ 所有 9 个必需库已就绪
✓ 中文字体已加载:NotoSansCJK-Regular.ttc
✓ 环境就绪 — 现在可以跑下面的代码单元格


## 学习目标

- 搞懂质量因子在赌什么:挑真正能持续高效赚钱的好公司,赌好公司长期跑赢烂公司
- 彻底吃透净资产收益率ROE:它就是「1 块钱本钱一年赚回几毛」,是衡量赚钱效率最核心的指标
- 学会分辨真假高ROE:用杜邦拆解看一家公司的高ROE是靠真本事,还是靠高杠杆硬撑
- 掌握「现金流照妖镜」:用经营现金流和净利润的比值,识破利润全是应收账款的纸面富贵
- 理解质量+价值的经典组合:既要好公司、又要不贵,这是巴菲特说的「用合理价格买伟大公司」

## 历史背景:老陈专挑高ROE的票,买了一只ROE体面的建筑股,业绩年年增长,可股价就是不涨、分红也抠,3 年后才明白:利润全是工地上收不回来的欠条

老陈是个看财报的老股民,信一条:ROE高的就是好公司。他翻数据挑中一只大建筑公司,净资产收益率年年有89%,净利润季季为正、还连年增长,看着特别扎实,重仓买入。可奇怪的是,这公司业绩明明在涨,股价却3 年原地踏步,分红也少得可怜。老陈百思不解,去请教一个做财务的朋友。朋友看完直摇头,说你只看了利润表,没看现金流量表。这公司利润是赚了,可钱呢?它给政府、给开发商修路盖楼,活干完了、利润记上了,可工程款要分好多年才慢慢收回来,甚至有的根本收不回来,全挂在「应收账款」里。账面上每年赚几十亿,经营现金流却常年是负的,等于赚的是一堆白条,口袋里根本没进钱。没现金,拿什么给你分红、拿什么扩大再生产、拿什么推高股价?老陈这才明白,光看ROE体面远远不够,得问一句:这利润,到底有没有变成真金白银落进口袋。



## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. 1. 质量因子:说白了就是挑「真正会赚钱」的好公司

前面价值挑便宜、成长挑长得快,质量因子换了个角度:不管贵不贵、长得快不快,先问这家公司「赚钱的本事硬不硬」。它专挑那些常年能用较少的本钱赚回较多利润、而且赚得稳、赚得久的公司。背后的逻辑是巴菲特那套:长期看,一家公司的股价回报,会越来越靠近它真实的赚钱能力。烂公司迟早现原形,好公司能穿越牛熊。质量因子就是把「好公司」这个模糊的感觉,变成一把能打分排名的尺子。


### 2. 2. 净资产收益率ROE:1 块钱本钱,一年赚回几毛

衡量「会不会赚钱」,最核心的一把尺子叫净资产收益率,缩写ROE。它的算法很简单:净利润除以净资产。净资产就是公司的家底(自己的钱),净利润是一年赚的钱。举个例子,一家公司家底100块,一年赚了20块,ROE就是20%,意思是「每1 块钱本钱,一年能赚回两毛」。ROE越高,说明用钱的效率越高、赚钱本事越强。常年ROE能稳在15%以上的,基本都是各行各业的尖子生。质量因子最常用的,就是给高ROE的票打高分。


### 3. 3. 高ROE有真有假:用杜邦拆解照一照

但ROE高,不一定是真本事。有个经典的拆解方法叫杜邦分析,把ROE拆成3 块:卖东西的利润率、资产周转的快慢、还有借钱的杠杆。前两个是真本事(产品赚钱、运营高效),第三个杠杆却是把双刃剑。有的公司ROE看着高,其实是借了一屁股债撑出来的——一旦行情逆转、利率上升,高杠杆会反过来把它拖垮。所以看到高ROE,要多问一句:它是靠产品和效率赚的,还是靠胆大加杠杆赌出来的。


### 4. 4. 现金流照妖镜:识破「纸面富贵」

比杠杆更隐蔽的坑,是利润压根没变成现金。有一个特别好用的指标,叫经营现金净流量和净利润的比值。正常的好公司,这个比值应该接近1,意思是「每赚1块钱利润,就有差不多1块钱现金落袋」。可有些公司,尤其是建筑、地产这类,账面利润年年为正、ROE也体面,但活干完了钱要好多年才收回来,甚至收不回来,全堆在应收账款里。它们这个比值常年远低于1、甚至是负数。这就叫纸面富贵:账上赚钱、口袋没钱。这面照妖镜,能帮你避开老陈踩的坑。


### 5. 5. 质量+价值:用合理价格买伟大公司

质量因子单用也有毛病:好公司人人都爱,容易被买得很贵,贵了再好也难赚钱。所以最经典的搭配,是质量和价值结合——既要是真能赚钱的好公司,又要价格不离谱,这就是巴菲特和芒格说的「用合理的价格买入一家伟大的公司,远胜于用便宜的价格买入一家平庸的公司」。前面的价值因子负责「便宜」,这一节的质量因子负责「好」,两个一叠加,选股的胜率会高很多。


## 实操:质量因子:高 ROE 分层回测 + 纸面富贵(经营现金流照妖镜)

下面这段代码跟视频里讲解的 highlights 是一致的,可以**直接 Run All** 看结果。

**依赖安装:**
```bash
pip install pandas numpy matplotlib yfinance akshare statsmodels
```


In [2]:
# day_070_quality_factor.py — 质量因子:高 ROE 分层回测 + 纸面富贵(经营现金流照妖镜)
# 真名上屏:baostock / query_profit_data → roeAvg(净资产收益率=质量因子核心) / query_cash_flow_data → CFOToNP(经营现金净流量÷净利润) / pubDate(披露日)PIT / 月末调仓分层
# 核心类比:ROE=「一块钱本钱一年能赚回几毛」,质量好的公司常年高 ROE;CFOToNP≈1 说明赚的利润有真金白银进账,远低于 1 = 利润是应收账款堆出来的「纸面富贵」(账上赚钱、口袋没钱)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import baostock as bs


def _data_path(_name):
    # 铁律62:data/ 放在 notebook 文件夹里。仓库根(run_lab)存取 out/notebook/data/;
    # 原版 notebook 在 out/notebook/ 用 data/;中国版在 out/notebook/cn/ 用 ../data/
    from pathlib import Path as _P
    _here = _P.cwd()
    for _b in [_here / 'data', _here / '..' / 'data', _here / 'out' / 'notebook' / 'data', _here / '..' / '..' / 'data', _here / '..' / '..' / '..' / 'data']:
        if (_b / _name).exists():
            return str(_b / _name)
    if (_here / 'out' / 'notebook').exists():
        _t = _here / 'out' / 'notebook' / 'data'
    elif _here.name == 'cn':
        _t = _here / '..' / 'data'
    else:
        _t = _here / 'data'
    _t.mkdir(parents=True, exist_ok=True)
    return str(_t / _name)


pd.set_option('display.width', 160)
plt.rcParams['axes.unicode_minus'] = False

# ==== 标的池:5 只高质量真本事(消费,常年高 ROE + 经营现金流充沛 CFOToNP≈1) ====
#               5 只纸面富贵嫌疑(建筑/地产,ROE 体面但靠杠杆/应收,经营现金流薄 CFOToNP 低)
CONSUMER = {                                           # 高质量真本事:利润是真现金
    '洋河股份': 'sz.002304', '苏泊尔': 'sz.002032', '涪陵榨菜': 'sz.002507',
    '海大集团': 'sz.002311', '重庆啤酒': 'sh.600132',
}
BUILDER = {                                            # 纸面富贵嫌疑:净利常年为正、ROE 体面,但现金没进来
    '中国建筑': 'sh.601668', '中国中冶': 'sh.601618', '中国铁建': 'sh.601186',
    '中国交建': 'sh.601800', '绿地控股': 'sh.600606',
}
STOCKS = {**CONSUMER, **BUILDER}
START, END = '2018-01-01', '2024-12-31'
CSV_PX = _data_path('D070_quality_close.csv')          # 前复权收盘(算策略真实收益)
CSV_FIN = _data_path('D070_quality_fin.csv')           # 季度财报:pubDate/statDate/roeAvg(质量)/netProfit/CFOToNP(纸面富贵照妖镜)


# ==== 0. 拉数据:前复权收盘 + 季度 roeAvg / netProfit / CFOToNP 及其披露日 ====
def _q(fn, **kw):
    # baostock 偶发瞬时报错(尤其并发/调用多时)→ 最多 3 次,失败就重新登录再试,保证不漏数据
    rs = fn(**kw)
    for _ in range(3):
        if rs.error_code == '0':
            return rs
        try:
            bs.logout()
        except Exception:
            pass
        bs.login()
        rs = fn(**kw)
    return rs


def _fetch():
    bs.login()
    px = {}
    fin_rows = []
    for name, code in STOCKS.items():
        # 前复权日线收盘(adjustflag='2' 前复权,算策略真实收益)
        rs = _q(bs.query_history_k_data_plus, code=code, fields='date,close',
                start_date=START, end_date=END, frequency='d', adjustflag='2')
        r = []
        while rs.error_code == '0' and rs.next():
            r.append(rs.get_row_data())
        s = pd.DataFrame(r, columns=['date', 'close'])
        s['close'] = pd.to_numeric(s['close'], errors='coerce')
        px[name] = s.set_index('date')['close']
        # 季度财报循环:盈利表(roeAvg/netProfit)+ 现金流量表(CFOToNP)
        for y in range(2017, 2025):
            for q in range(1, 5):
                # 盈利能力:净资产收益率 roeAvg(质量因子核心)+ 净利润 netProfit + 披露日 pubDate
                rp = _q(bs.query_profit_data, code=code, year=y, quarter=q)
                roe, netp, pub_p, stat_p = '', '', '', ''
                while rp.error_code == '0' and rp.next():
                    p = rp.get_row_data()   # code,pubDate,statDate,roeAvg,npMargin,gpMargin,netProfit,epsTTM,MBRevenue,totalShare,liqaShare
                    pub_p, stat_p, roe, netp = p[1], p[2], p[3], p[6]
                # 现金流量:经营现金净流量÷净利润 CFOToNP(纸面富贵照妖镜)
                rc = _q(bs.query_cash_flow_data, code=code, year=y, quarter=q)
                cfo2np, pub_c, stat_c = '', '', ''
                while rc.error_code == '0' and rc.next():
                    c = rc.get_row_data()   # code,pubDate,statDate,CAToAsset,NCAToAsset,tangibleAssetToAsset,ebitToInterest,CFOToOR,CFOToNP,CFOToGr
                    pub_c, stat_c, cfo2np = c[1], c[2], c[8]
                fin_rows.append({
                    'name': name,
                    'pubDate': pub_p or pub_c,
                    'statDate': stat_p or stat_c,
                    'roeAvg': roe,
                    'netProfit': netp,
                    'CFOToNP': cfo2np,
                })
    bs.logout()
    return pd.DataFrame(px), pd.DataFrame(fin_rows)


if os.path.exists(CSV_PX) and os.path.exists(CSV_FIN):
    print('[数据] 从本地 CSV 读取 收盘 + 季度财报(roeAvg/CFOToNP)')
    px = pd.read_csv(CSV_PX, parse_dates=['date']).set_index('date')
    fin = pd.read_csv(CSV_FIN)
else:
    print('[数据] baostock 拉取 10 只票 前复权收盘 + 季度 roeAvg/netProfit/CFOToNP ...')
    px, fin = _fetch()
    px.index.name = 'date'
    px.to_csv(CSV_PX)                     # 拉完保存 CSV(铁律62:load 优先 + fetch 兜底 + 自动保存)
    fin.to_csv(CSV_FIN, index=False)
    print('[数据] 已存 CSV ->', CSV_PX, '/', CSV_FIN)

px.index = pd.to_datetime(px.index)
px = px[[n for n in STOCKS if n in px.columns]]        # 固定列顺序(消费 5 + 建筑 5)

fin['pubDate'] = pd.to_datetime(fin['pubDate'], errors='coerce')
fin['statDate'] = pd.to_datetime(fin['statDate'], errors='coerce')
for col in ['roeAvg', 'netProfit', 'CFOToNP']:
    fin[col] = pd.to_numeric(fin[col], errors='coerce')
fin = fin.dropna(subset=['pubDate', 'statDate'])

print('\n==== 质量因子:高 ROE 分层回测 + 纸面富贵(经营现金流照妖镜)====')
print('标的池 %d 只(消费高质量 %d / 建筑地产嫌疑 %d) · %s ~ %s' % (
    len(STOCKS), len(CONSUMER), len(BUILDER), px.index[0].date(), px.index[-1].date()))


def maxdd(equity):
    # 最大回撤(%):从历史最高点跌下来最深的那一下
    e = pd.Series(equity).dropna()
    if len(e) < 2:
        return 0.0
    peak = e.cummax()
    return float((e / peak - 1.0).min() * 100)


def ann_return(equity, n_months):
    # 年化收益(%):把总收益按月数折算成每年
    e = pd.Series(equity).dropna()
    if len(e) < 2 or n_months <= 0:
        return 0.0
    yrs = n_months / 12.0
    base = e.iloc[-1]
    if base <= 0:
        return float('nan')
    return float((base ** (1.0 / yrs) - 1.0) * 100)


# ==== 1. 质量信号 PIT:roeAvg 挂到披露日 pubDate,ffill 到月末(每月当时已知的最新 ROE)====
monthly = px.resample('ME').last()                     # 月末收盘
mret = monthly.pct_change().shift(-1)                  # 本月末建仓 -> 下月末的收益


def signal_panel(date_col, value_col):
    # 严格 point-in-time:财报值挂到「披露日」而非「报告期末」,再 ffill 到每个月末(避免未来函数)
    panel = {}
    for name in STOCKS:
        sub = fin[fin['name'] == name].dropna(subset=[value_col]).sort_values(date_col)
        if len(sub) == 0:
            panel[name] = pd.Series(np.nan, index=monthly.index)
            continue
        ser = pd.Series(sub[value_col].values, index=sub[date_col])
        ser = ser[~ser.index.duplicated(keep='last')].sort_index()
        panel[name] = ser.reindex(monthly.index, method='ffill')
    return pd.DataFrame(panel)


sig_roe = signal_panel('pubDate', 'roeAvg')            # PIT 质量信号:净资产收益率 roeAvg


# ==== 2. 月末调仓:高质量(高ROE)组 / 低质量(低ROE)组 / 基准等权 三条净值 ====
def backtest(mode):
    # mode: 'high'=高ROE质量组 / 'low'=低ROE质量组 / 'base'=等权全部
    eq, dates = [1.0], [monthly.index[0]]
    for i in range(len(monthly.index) - 1):
        t = monthly.index[i]
        v = sig_roe.loc[t].dropna()                    # 当月已知的各票最新 ROE
        r_next = mret.loc[t]
        n = len(v)
        if n >= 2 and mode != 'base':
            k = max(2, n // 2)                          # 一半高质量 / 一半低质量
            k = min(k, n)
            if mode == 'high':
                sel = v.nlargest(k).index              # 高质量:ROE 最高的一组
            else:
                sel = v.nsmallest(k).index             # 低质量:ROE 最低的一组
            r = r_next[sel].mean()
        else:
            r = r_next.mean()                          # 基准:全池等权
        r = 0.0 if pd.isna(r) else r
        eq.append(eq[-1] * (1 + r))
        dates.append(monthly.index[i + 1])
    return pd.Series(eq, index=dates)


eq_high = backtest('high')
eq_low = backtest('low')
eq_base = backtest('base')
n_months = len(monthly.index) - 1

print('\n[高质量(高ROE)组 / 低质量(低ROE)组 / 基准等权 — 同一段 %d 个月]' % n_months)
res = {}
for tag, eq in [('高质量组', eq_high), ('低质量组', eq_low), ('基准等权', eq_base)]:
    tot = (eq.iloc[-1] - 1) * 100
    ann = ann_return(eq.values, n_months)
    dd = maxdd(eq.values)
    res[tag] = (tot, ann, dd, eq.iloc[-1])
    print('%s  总收益 %+.1f%%   年化 %+.1f%%   最大回撤 %.1f%%   终值 %.3f' % (tag, tot, ann, dd, eq.iloc[-1]))
_q_premium = res['高质量组'][0] - res['低质量组'][0]
print('质量溢价 = %+.1f 个百分点(高质量组 − 低质量组总收益)' % _q_premium)


# ==== 3. 纸面富贵:CFOToNP(经营现金净流量÷净利润)照妖镜 ====
# CFOToNP≈1:每赚 1 元利润就有约 1 元现金进账(真金白银);CFOToNP 远低于 1:利润靠应收账款堆出来,现金没进口袋
cfo_mean = {}
roe_mean = {}
netp_pos_ratio = {}                                    # 净利润为正的季度占比(确认「账面是赚钱的」)
for name in STOCKS:
    sub = fin[fin['name'] == name]
    cser = sub['CFOToNP'].dropna()
    rser = sub['roeAvg'].dropna()
    nser = sub['netProfit'].dropna()
    cfo_mean[name] = float(cser.mean()) if len(cser) else np.nan
    roe_mean[name] = float(rser.mean()) if len(rser) else np.nan
    netp_pos_ratio[name] = float((nser > 0).mean()) if len(nser) else np.nan

quality_tbl = pd.DataFrame({
    '组别': pd.Series({n: ('消费高质量' if n in CONSUMER else '建筑地产嫌疑') for n in STOCKS}),
    '常年ROE%': pd.Series(roe_mean) * 100,
    'CFOToNP': pd.Series(cfo_mean),
    '净利为正季占比%': pd.Series(netp_pos_ratio) * 100,
}).round(2)
quality_tbl = quality_tbl.sort_values('CFOToNP', ascending=False)
print('\n[质量全景:常年 ROE(质量)+ 现金含金量 CFOToNP(纸面富贵照妖镜)]')
print(quality_tbl.to_string())

# 纸面富贵实例:建筑地产组里挑「净利常年为正、可 CFOToNP 最低」的一只(账面赚钱、现金没进来)
builder_tbl = quality_tbl[quality_tbl['组别'] == '建筑地产嫌疑'].dropna(subset=['CFOToNP'])
paper_name = builder_tbl['CFOToNP'].idxmin()           # 现金含金量最差的建筑票 = 纸面富贵代表
# 真现金对照:消费组里挑 CFOToNP 最接近 1(利润有真金白银)的一只
consumer_tbl = quality_tbl[quality_tbl['组别'] == '消费高质量'].dropna(subset=['CFOToNP'])
real_name = (consumer_tbl['CFOToNP'] - 1.0).abs().idxmin()   # 现金/利润最接近 1 = 真现金代表
_paper_cfo = quality_tbl.loc[paper_name, 'CFOToNP']
_real_cfo = quality_tbl.loc[real_name, 'CFOToNP']
_paper_roe = quality_tbl.loc[paper_name, '常年ROE%']
_real_roe = quality_tbl.loc[real_name, '常年ROE%']
print('\n[纸面富贵实例] %s(建筑地产):常年 ROE %.1f%% 看着体面、净利几乎季季为正,可 CFOToNP 均值仅 %.2f' % (
    paper_name, _paper_roe, _paper_cfo))
print('  -> 每赚 1 元利润只收回约 %.2f 元现金,利润大半堆在应收账款里 = 纸面富贵' % _paper_cfo)
print('[真现金对照] %s(消费):常年 ROE %.1f%%,CFOToNP 均值 %.2f ≈ 1,利润是真金白银落袋' % (
    real_name, _real_roe, _real_cfo))
print('  -> 现金含金量差距 = %.2f vs %.2f,相差 %.0f%%' % (
    _real_cfo, _paper_cfo, (_real_cfo - _paper_cfo) / abs(_paper_cfo) * 100 if _paper_cfo else float('nan')))

# 两组现金含金量均值对比(收尾结论用)
cfo_consumer = consumer_tbl['CFOToNP'].mean()
cfo_builder = builder_tbl['CFOToNP'].mean()
print('\n[两组现金含金量对比] 消费组 CFOToNP 均值 %.2f vs 建筑地产组 %.2f —— 消费真现金、建筑纸面富贵' % (
    cfo_consumer, cfo_builder))


# ==== 4. 三张图 ====
# 图1:高质量组 vs 低质量组 vs 基准 三条净值曲线(质量因子超额)
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(eq_high.index, eq_high.values, lw=2, c='#c62828', label='高质量(高ROE)组')
ax.plot(eq_low.index, eq_low.values, lw=2, c='#1565c0', label='低质量(低ROE)组')
ax.plot(eq_base.index, eq_base.values, lw=1.6, c='#777', ls='--', label='基准等权')
ax.axhline(1.0, c='k', lw=.8)
for eq, c in [(eq_high, '#c62828'), (eq_low, '#1565c0'), (eq_base, '#777')]:
    ax.text(eq.index[-1], eq.iloc[-1], ' %.2f' % eq.iloc[-1], color=c, fontweight='bold', va='center')
_winner = '高质量(高ROE)组' if res['高质量组'][0] > res['低质量组'][0] else '低质量(低ROE)组'
ax.set_title('质量因子:高ROE组 vs 低ROE组 vs 基准(2018-2024),%s跑赢、质量溢价 %+.0f 个百分点(终值已标注)' % (
    _winner, _q_premium))
ax.set_ylabel('净值(起点=1)')
ax.legend(loc='upper left')
ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig('chart_01.png', dpi=110)
plt.close()

# 图2:纸面富贵照妖镜 —— 建筑票 vs 消费票 的 CFOToNP 时间序列(建筑常年低于1,消费贴着1)
fig, ax = plt.subplots(figsize=(11, 6))
paper_ser = fin[fin['name'] == paper_name].dropna(subset=['CFOToNP', 'statDate']).sort_values('statDate')
real_ser = fin[fin['name'] == real_name].dropna(subset=['CFOToNP', 'statDate']).sort_values('statDate')
ax.plot(paper_ser['statDate'], paper_ser['CFOToNP'], lw=2, marker='o', ms=4, c='#1565c0',
        label='%s(建筑地产·纸面富贵)CFOToNP' % paper_name)
ax.plot(real_ser['statDate'], real_ser['CFOToNP'], lw=2, marker='s', ms=4, c='#c62828',
        label='%s(消费·真现金)CFOToNP' % real_name)
ax.axhline(1.0, c='k', ls=':', lw=1.2)
ax.text(ax.get_xlim()[1], 1.0, ' =1:利润全变现金', va='center', fontsize=9, color='#333')
ax.axhline(0.0, c='#999', lw=.8)
ax.set_title('纸面富贵照妖镜:%s 现金含金量常年只有 %.2f(账面赚钱、现金没进来) vs %s 贴着 1(真金白银)' % (
    paper_name, _paper_cfo, real_name))
ax.set_ylabel('CFOToNP(经营现金净流量 ÷ 净利润)')
ax.legend(loc='upper left')
ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig('chart_02.png', dpi=110)
plt.close()

# 图3:各票现金含金量 CFOToNP 横向柱状排行,红=消费高质量 / 蓝=建筑地产纸面富贵,标注数值
fig, ax = plt.subplots(figsize=(11, 6))
plot_tbl = quality_tbl.dropna(subset=['CFOToNP']).sort_values('CFOToNP')
colors = ['#c62828' if g == '消费高质量' else '#1565c0' for g in plot_tbl['组别']]
ax.barh(plot_tbl.index, plot_tbl['CFOToNP'], color=colors)
for i, (idx, v) in enumerate(plot_tbl['CFOToNP'].items()):
    ax.text(v + (0.03 if v >= 0 else -0.03), i, '%.2f' % v, va='center',
            ha='left' if v >= 0 else 'right', fontweight='bold', fontsize=9)
ax.axvline(1.0, c='k', ls=':', lw=1.2)
ax.text(1.0, len(plot_tbl) - 0.4, ' =1 真金白银线', fontsize=9, color='#333')
ax.set_title('现金含金量排行榜 CFOToNP:消费(红)贴近/超过 1=真现金,建筑地产(蓝)常年偏低=纸面富贵')
ax.set_xlabel('CFOToNP(经营现金净流量 ÷ 净利润,越接近 1 利润越实)')
ax.grid(alpha=.3, axis='x')
fig.tight_layout()
fig.savefig('chart_03.png', dpi=110)
plt.close()

print('\n[done] 质量因子高ROE分层回测 + 纸面富贵实证完成 —— 3 图已出')
print('[结论速记] 高质量组总收益 %+.0f%% vs 低质量组 %+.0f%%(质量溢价 %+.0f pct);纸面富贵代表 %s CFOToNP %.2f vs 真现金代表 %s %.2f' % (
    res['高质量组'][0], res['低质量组'][0], _q_premium, paper_name, _paper_cfo, real_name, _real_cfo))


[数据] 从本地 CSV 读取 收盘 + 季度财报(roeAvg/CFOToNP)

==== 质量因子:高 ROE 分层回测 + 纸面富贵(经营现金流照妖镜)====
标的池 10 只(消费高质量 5 / 建筑地产嫌疑 5) · 2018-01-02 ~ 2024-12-31

[高质量(高ROE)组 / 低质量(低ROE)组 / 基准等权 — 同一段 83 个月]
高质量组  总收益 +102.6%   年化 +10.7%   最大回撤 -47.6%   终值 2.026
低质量组  总收益 -14.2%   年化 -2.2%   最大回撤 -33.0%   终值 0.858
基准等权  总收益 +41.0%   年化 +5.1%   最大回撤 -28.8%   终值 1.410
质量溢价 = +116.9 个百分点(高质量组 − 低质量组总收益)

[质量全景:常年 ROE(质量)+ 现金含金量 CFOToNP(纸面富贵照妖镜)]
          组别  常年ROE%  CFOToNP  净利为正季占比%
重庆啤酒   消费高质量   41.95     1.79    100.00
苏泊尔    消费高质量   17.37     0.84    100.00
涪陵榨菜   消费高质量   11.84     0.73    100.00
海大集团   消费高质量   11.50     0.51    100.00
洋河股份   消费高质量   16.59     0.40    100.00
绿地控股  建筑地产嫌疑    6.65    -0.47     93.75
中国中冶  建筑地产嫌疑    4.71    -2.06    100.00
中国建筑  建筑地产嫌疑    9.44    -2.50    100.00
中国交建  建筑地产嫌疑    5.07    -3.23    100.00
中国铁建  建筑地产嫌疑    5.87    -3.29    100.00

[纸面富贵实例] 中国铁建(建筑地产):常年 ROE 5.9% 看着体面、净利几乎季季为正,可 CFOToNP 均值仅 -3.29
  -> 每赚 1 元利润只收回约 -3.29 元现金,利润大半堆在应收账款里 = 纸面富贵
[真现金对照] 苏泊尔(消费):常年 RO

## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| 质量溢价·好公司持续跑赢 | N/A | 2018到2024这7 年,高质量(高ROE)组总收益+102.6%、终值2.026,大幅跑赢低质量(低ROE)组-14.2%、终值0.858(7 年还亏了),也跑赢基准等权+41%。质量溢价+116.9个点。常年高ROE的好公司一路爬升、低ROE的持续跑输,质量因子在A股很扎实。 |
| 纸面富贵·中国铁建 | sh.601186 | 中国铁建常年净资产收益率5.9%看着体面、净利润季季为正,可它的经营现金流÷净利润(CFOToNP)均值只有-3.29:每赚1元账面利润,经营现金不仅没进来、反而流出约3.29元。利润全堆在工程应收账款里,账上赚钱、口袋没钱,这就是典型的纸面富贵——光看ROE根本看不出来。 |
| 真金白银·苏泊尔 | sz.002032 | 苏泊尔常年ROE17.4%,经营现金流÷净利润0.84≈1,每赚1块钱利润就有约1块钱现金落袋。它是消费高质量的代表:赚钱效率高、而且赚的是真金白银。跟中国铁建一对照,同样体面的报表,现金含金量从0.84到-3.29、相差超过4倍,质量成色天壤之别。 |
| 两组分化·现金含金量 | N/A | 10只票按现金含金量CFOToNP排榜,泾渭分明:消费组(重庆啤酒1.79、苏泊尔0.84、涪陵榨菜0.73、海大0.51、洋河0.40)全部为正,均值0.85;建筑地产组(绿地-0.47、中冶-2.06、中建-2.50、中交-3.23、中铁建-3.29)全部为负,均值-2.31。质量因子必须叠加现金流验真,否则会把纸面富贵当好公司。 |


## 常见坑

### ⚠ 01. 只看ROE不看现金流,踩进纸面富贵

光盯着ROE高就买,很容易买到老陈那种利润全是应收账款的公司。账面赚钱、口袋没钱,这种「好公司」分红抠门、股价不涨。必须用经营现金流和净利润的比值验真,接近1才放心。

### ⚠ 02. 高ROE其实是高杠杆撑出来的

有些公司ROE看着漂亮,其实是借了大量债把杠杆加上去硬撑的。这种高ROE在顺风时风光,一旦行业下行、利率上升,杠杆反噬,跌起来比谁都狠。要用杜邦拆解看清ROE的成色。

### ⚠ 03. 把一年的高ROE当成持续能力

某一年ROE突然很高,可能只是卖了块地、收了笔一次性的钱。真正的质量,是常年稳定的高ROE,不是某一年的昙花一现。要看连续好几年的ROE是不是都稳得住。

### ⚠ 04. 不看行业差异,横向硬比

银行靠高杠杆ROE天然偏高,科技公司轻资产ROE也容易高,重工业则普遍偏低。拿不同行业的ROE直接横向比,会误判。质量因子要在可比的范围里比,或者结合行业看。

### ⚠ 05. 好公司买在太贵的位置

质量因子最大的风险不是选错公司,而是买太贵。一家伟大的公司,如果你在它最被追捧、估值最高的时候冲进去,也得忍受很长时间的回调。好公司也要等一个不离谱的价格,这就是质量必须配合价值的原因。

## 实战 SOP · 用质量因子选股 — 几条铁规矩

1. 先想清楚质量因子在赌什么:赌真正会赚钱的好公司长期跑赢烂公司,它是挑一篮子高质量的票,不是预测短期涨跌。
2. 核心尺子用ROE:挑常年(至少35 年)稳定高ROE的公司,一年的高ROE不算数,要看持续性。
3. 必用现金流验真:经营现金流和净利润的比值接近1才是真赚钱,远低于1或为负就是纸面富贵,坚决回避。
4. 用杜邦拆解看成色:警惕靠高杠杆撑出来的高ROE,优先选靠产品利润率和运营效率赚钱的公司。
5. 质量必须配合价格:再好的公司也别买在最贵的时候,质量+价值一起用,用合理价格买好公司。

> 把这段打印贴在你电脑边,执行 1000 次它会回报你。

## 总结 · 你应该带走的

2. 质量因子=挑真正会赚钱的好公司,赌好公司长期跑赢烂公司,是巴菲特最看重的一把尺子。
3. 核心尺子是净资产收益率ROE=净利润÷净资产,说白了就是1 块钱本钱一年赚回几毛,常年稳在15%以上的是尖子生。
4. 质量溢价实测:2018-2024高质量(高ROE)组+102.6%大幅跑赢低质量组-14.2%(7 年还亏),溢价+116.9个点。
5. 高ROE有真有假:有的靠高杠杆撑、有的利润全是应收账款的纸面富贵,必须用现金流验真。
6. 现金含金量CFOToNP=经营现金流÷净利润,接近1才是真现金;消费组0.85≈真金白银,建筑地产组-2.31是纸面富贵(中国铁建-3.29)。
7. 质量要配合价值用:好公司也别买太贵,用合理价格买伟大公司,质量负责好、价值负责便宜。

## 自测题

**Q1.** 净资产收益率ROE到底在衡量什么?请用「1 块钱本钱赚回几毛」这个说法,讲清楚它为什么能代表一家公司的赚钱能力。

**Q2.** 老陈买的建筑股ROE体面、利润年年增长,为什么还是踩了坑?「纸面富贵」是怎么回事,用什么指标能提前照出来?

**Q3.** 都说「质量+价值」是经典组合,为什么质量因子单用还不够、非要配上价值因子?用巴菲特那句「合理价格买伟大公司」解释。

把答案写下来,3 天后再回看。

## 下一节预告

**Day 071 · 动量因子详解** (Momentum)

这一节我们用ROE挑会赚钱的好公司,还用现金流照妖镜识破了纸面富贵。下一节我们换一个完全不看基本面、只看价格走势的视角:动量因子。它信奉「强者恒强」,专门追那些最近一直在涨的票,赌它们继续涨。但你猜怎么着,这个看起来最简单的策略,藏着一个最凶险的陷阱——动量崩盘,转折点上能让你一夜回到解放前。

## 推荐阅读

- Robert Novy-Marx《The Other Side of Value: The Gross Profitability Premium》(2013/JFE) — 用毛利润率衡量质量,证明「质量」和「便宜」一样能带来超额收益的开创性论文
- Clifford Asness, Andrea Frazzini, Lasse Pedersen《Quality Minus Junk》(2019/RoF) — AQM 量化巨头给「质量」下了严格定义(盈利、成长、安全、分红),好公司减烂公司长期赚钱
- Richard Sloan《Do Stock Prices Fully Reflect Information in Accruals and Cash Flows?》(1996/AR) — 应计利润异象奠基:利润里现金少、应计多的公司未来跑输,正是「纸面富贵」的学术源头
- Joseph Piotroski《Value Investing: The Use of Historical Financial Statement Information》(2000/JAR) — 著名的 F-Score 九项打分,教你用财报指标把好公司从便宜票里筛出来
- 罗伯特·哈格斯特朗《巴菲特之道》(2014/中信出版社) — 用大白话讲透巴菲特如何用ROE、现金流、护城河挑伟大公司,散户读质量投资的最佳入门